<a href="https://colab.research.google.com/github/dowes48/A2E_polars/blob/main/A2E_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast A2E Using Polars - Google Colab Session

When you run one of the following Colab code cells for the first time, Google will warn you that this is not a Google notebook. Choose the option "Run Anyway".

For example, **Click on the arrow in the next cell** to determine what version of Python Google is using, then choose the option "Run Anyway".  

*one can also run a code cell by placing the cursor in the cell and keying "Shift-Enter"*

In [ ]:
!python --version

Python 3.12.12 was released in 10/2025 as a security fix for version 3.12 which was first released in 10/2023. Since the current stable version of Python is 3.14, one may occasionally find that the latest Python expressions may not work in Colab. See https://docs.python.org/release/3.12.12/ for version specific documentation.

*Colab can also be configured to run R code.*

---

I am using GitHub to store this Notebook and its associated data files. If you got this far, you already have a Colab copy of my notebook. Now you need to do next is to **click the arrow in the next cell** to make a temporary copy of my entire GitHub repository including the data files that we will want to use.

In [ ]:
!git clone "https://github.com/dowes48/A2E_polars"

In [ ]:
# run this only if you plan to use the mega dataset
# in that case, remove the hashtag from the next line

# !unzip "/content/A2E_polars/data/a2e_mega_data_35.zip" -d "/content/A2E_polars/data"


After the cloning has finished, click on the File folder icon to the left. A folder for A2E_polars should appear. Under this, you will find my data folder.

---

This is a "Text" cell - actually created as Markdown language. Colab is very nice to use for **Markdown** because while the Markdown special characters are entered in the left-hand side of this cell, the right-hand side shows how the text will appear to the reader. Double-click on this cell to enter editing mode and you will see what I mean.

---

The following following cell *imports* the libraries that will be used in this notebook. The cells that follow depend on the libraries' code, so make certain this is run first. In general, run cells in the given sequence.

In [ ]:
import os                                # basic file handling functions
import polars as pl                      # polars, a DataFrame library for manipulating structured data
from datetime import timedelta           # a single datatype needed for calculations
import time

# configure polars
with pl.Config():
    tbl_cell_numeric_alignment="RIGHT",
    float_precision=5,
    tbl_rows=30,
    tbl_cols=100
    fmt_str_lengths=200,
    tbl_width_chars=1000,


In [ ]:
# read the data file into a polars dataframe ('pl' is shorthand for polars)
df = pl.read_csv(r'A2E_polars/data/HRA_data.csv', try_parse_dates = True)

# you can choose to run the 10-fold expansion dataset instead
# df = pl.read_csv(r'/content/A2E_polars/data/big_data_test.csv', try_parse_dates = True)

# go back to the unzip cell if you want to runn the 100-fold expansion
# df = pl.read_csv(r'/content/A2E_polars/data/a2e_mega_data_35.csv', try_parse_dates = True)

# Study Dataset
* "HRA_data/csv" is the original dataset
* "big_data_test.csv" is a 10-fold expansion
* "a2e_mega_data_35" is a 100-fold expansion

Each row of the 24,528 | 245,280 | 2,452,800 rows in the dataset represent a subjects' “Health Risk Appraisals” done at 9 different sites starting in January 1998 with a study end date of 6/30/2008. Since this is synthetic dataset, mortality follow-up is 100% complete and error-free.

Felds (columns) are as follows:
*   **UID** is a unique identiﬁer for each individual
*   **examD** is the date of examination (ﬁrst exam was 1/2/1998)
*   **exitD** is either the date of death or the study end date (6/30/2008)
*   **vs** is the vital status: [“alive”, “dead”]
*   **age** is the age at time of exam: 65–84
*   **dob** is the date of birth for the individual
*   **sex** is ['F', 'M']
*   **dxGrp** will be one of eight categories: [“healthy”, “cancer”, “cardiac”, “diabetes”, “neuro”, “pulm”, “renal”, “other”]
*   **examY** is the year of examination: 1998–2008
*   **site** is one of nine exam sites: [“AL”, “AZ”, “CA”, “FL”, “GA”, “LA”, “SC”, “TX”, ”VA"]
*   **mr** is the mortality ratio estimated by the examiner
*   **ageBand** is banded age at time of examination

A random ten row sample of the study data is displayed in the next cell


In [ ]:
pl.Config.set_tbl_cols(200)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(1000)

print(df.sample(n=10))

In the next cell, enumerated datatypes are created to provide better performance. Basically, data is stored as integers while they are shown as the original human-readable strings. This can be big.

Less important are the casting to smaller numerical datatypes.

In [ ]:
# rename exam year column
df = df.rename({'examY': 'year'})

# define Enumeration datatypes
enum_vs   = pl.Enum(["alive", "dead"])
enum_sex  = pl.Enum(["M", "F"])
enum_dx   = pl.Enum(["healthy", "cancer", "cardiac", "diabetes", "neuro", "pulm", "renal", "other"])
enum_site = pl.Enum(["AL", "AZ", "CA", "FL", "GA", "LA", "SC", "TX", "VA"])
enum_agebnd  = pl.Enum(["65 - 69", "70 - 74", "75 - 79", "80 - 84"])

# cast column values to defined datatypes
df = df.with_columns(
    pl.col("vs").cast(enum_vs),
    pl.col("sex").cast(enum_sex),
    pl.col("dxGrp").cast(enum_dx),
    pl.col("site").cast(enum_site),
    pl.col("ageBand").cast(enum_agebnd))

df = df.with_columns(
    pl.col("age").cast(pl.UInt8),
    pl.col("year").cast(pl.UInt16),
    pl.col("UID").cast(pl.UInt32),
    pl.col("mr").cast(pl.Float32))

Compare the column datatypes below with those in dataframe printout above.

In [ ]:
pl.Config.set_float_precision(2)
print(df.head(10))

The following sets up dataframe for expansion. Unnecessary columns are put aside in a "details" dataframe.

In [ ]:
# split off columns of "details"
df_details = df.select('UID', 'sex', 'mr', 'dxGrp', 'site', 'ageBand')
print("\n\ndetails dataframe 'df_details' variables will be re-joined asfter expansion")
print(df_details.head(10))

# keep columns needed for expansion
df = df.select('UID', 'examD', 'exitD', 'vs', 'age', 'year')

print("\nStudy dataframe 'df' now contains only variables needed for expansion")
print(df.head(10))

What follows is a determination of total time in study for each individual and the number of yearly intervals needed to represent each individual's time in study. It may appear simple, but it is not. Dates, times, time spans, etc. are all a challenge in data analysis.

In [ ]:
# tDelta constants to be used in exposure calculation
tDelta_1yr = timedelta(days=365, hours=6)
tDelta_1day  = timedelta(days=1)

# calculate total number of intervals for each subject
df = df.with_columns(totalIntvls = ((pl.col('exitD') - pl.col('examD') + tDelta_1day)/tDelta_1yr)
                     .ceil().cast(pl.Int32))

print(df.head(20))

## Now the clever part  (shown in three cells).
### First cell
1. It is computationally easy to create a column of identical values: each a list of numbers equal to the maximum number of intervals needed for this study.

    *lists are comma separated values within brackets* eg. [1, 2, 3, ...]




In [ ]:
# create temporary column of identical interval index lists
df = df.with_columns(idx_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

pl.Config.set_tbl_rows(200)
print(df.head(20))


### Second cell
2. It is also easy to create a column of slices from each list - how much is sliced is determined by the number of intervals required for each study participant.

    *each slice is also a list*


In [ ]:
# take slice of idx_list and store list in new column ‘intvl’
df = df.with_columns(intvl = pl.col('idx_list').list.slice(0, pl.col('totalIntvls')))
df = df.drop('idx_list')

pl.Config.set_tbl_rows(200)
print(df.head(20))

### Third cell
3. The dataframe.explode() function duplicates each row, with one row for each entry in the list stored in 'intvl'.



In [ ]:
#  explode to get desired tabular output
df = df.explode(pl.col('intvl'))

pl.Config.set_tbl_rows(200)
print(df.head(20))

In [ ]:
# create temporary column of offset years, then calculate ‘intvl_date’
df = df.with_columns(offsetYrs = (pl.col("intvl") - 1).cast(pl.String))
df = df.with_columns(intvl_date = pl.col('examD').dt.offset_by((pl.col('offsetYrs')) + "y"),
        intvl_age = pl.col('age') + pl.col('intvl') - 1)

print(df.head(20))


In [ ]:
# three important calculations: interval year, exposure, and actual
df = df.with_columns(
    # first the interval year (for acquiring qx)
    year = pl.col('intvl_date').dt.year(),

    # second the exposure
    persYrs = pl
        .when((pl.col('intvl_date').dt.offset_by('1y')) <= pl.col('exitD'))
        .then(pl.lit(1))
        .otherwise((pl.col('exitD') - pl.col('intvl_date') + tDelta_1day) / tDelta_1yr),

    # actual deaths
    actual = pl
        .when((pl.col('vs') == "dead") & (pl.col('intvl')
            == pl.col('totalIntvls')))
        .then(pl.lit(1))
        .otherwise(pl.lit(0)))


In [ ]:
pl.Config.set_float_precision(3)
print(df.head(20))

In [ ]:
# clean up and get ready for join
df = df.select('UID', 'vs', 'intvl', 'intvl_date', 'intvl_age',
                         'year', 'persYrs', 'actual')
df = df.rename({'intvl_age': 'age'})
df = df.join(df_details, on='UID', how='left')

pl.Config.set_float_precision(2)
print(df.head(20))

Load the mortality table.

In [ ]:
# load mortality dataframe
df_mort = pl.read_csv(r"A2E_polars/data/USA_both_1933-2023.psv",separator='|')
df_mort = df_mort.rename({'Year': 'year', 'Age': 'age'})
enum_sex  = pl.Enum(["M", "F"])
df_mort = df_mort.with_columns(pl.col('sex').cast(enum_sex))

Print a random sample of the mortality table.

In [ ]:
print("HMD: USA, both sexes, 1998-2023, Ages 65 to 84")
pl.Config.set_float_precision(5)
print(df_mort.filter((pl.col("year") >= 1998) & (pl.col('age')>=65) & (pl.col('age')<=84)).sample(n=15).sort('year', 'age'))

Now for the join.

In [ ]:
# join to get qx
df = df.join(df_mort, on=('year','age','sex'), how='left')


In [ ]:
pl.Config.set_float_precision(2)
print(df.head(20))

In [ ]:
# last calculation, expected deaths
df = df.with_columns(expected = pl.col('persYrs') * pl.col('qx') * pl.col('mr')).sort('UID', 'intvl')

df = df.select('UID', 'age', 'sex', 'mr', 'dxGrp', 'site', 'ageBand', 'intvl', 'intvl_date', 'year',
               'persYrs', 'actual', 'qx', 'expected').sort('UID', 'intvl')



In [ ]:
pl.Config.set_float_precision(4)
print(df.head(20))


Save the results to a csv file that can be opened in MS Excel where sophisticated pivot tables can be used for analysis. Remember to save as an xlsx file to retain any formatting.

We can do a quick pivot table here in polars for a simple check on the results.

In [ ]:
df.write_csv(r'A2E_polars/data/a2e_results_' + '.csv')
# double-clicking the csv file will open it in MS Excel

df_pivot = df.pivot(
    on='sex',
    index='ageBand',
    values=['actual', 'persYrs', 'expected'],
    aggregate_function='sum'
)

pl.Config.set_tbl_cell_numeric_alignment("RIGHT")
pl.Config.set_thousands_separator(',')
pl.Config.set_float_precision(1)

print(df_pivot)


**Remember** : the HRA dataset is 100% synthetic, so don't go crazy running Excel pivots.

For practice, I suggest you go back to an earlier cell and load a different dataset: "big_data_test.csv" or "a2e_mega_data_35.csv". Here the rows from HRA_data.csv have been "exploded" 10 or 100 times and new UID's were assigned. After making the necessary edits, it will be faster to choose "Run all" rather than stepping through the cells.

Feel free to make changes to adapt the Python/Polars code to your own dataset.